<h2 align="center"> Flair_&_FastText_Embedding<h2>

## Connect to the drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# check the connection
!ls "/content/gdrive/My Drive/Colab Notebooks/Flair Embedding"

char_mappings		 language_model_forward     pre_trained_muse.aren
corpus			 pre_trained.aren
language_model_backward  pre_trained_fasttext.aren


## Train our Flair embedding model

In [ ]:
# install flair
!pip3 install flair

In [ ]:
# import needed libraries
from flair.data import Dictionary
from flair.models import LanguageModel
from flair.trainers.language_model_trainer import LanguageModelTrainer, TextCorpus
import collections
import glob

In [ ]:
# our corpus rout
files = glob.glob('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/*.*')
files += glob.glob('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/*/*')
print(files)

['/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/valid.txt', '/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/test.txt', '/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/train/train_split_1']


In [ ]:
# make our char dictionary as the default does not have Arabic chars
char_dictionary: Dictionary = Dictionary()
counter = collections.Counter()
processed = 0
for file in files:
    print(file)
    with open(file, 'r', encoding='utf-8') as f:
        tokens = 0
        for line in f:
            processed += 1            
            chars = list(line)
            tokens += len(chars)
            counter.update(chars)

/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/valid.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/test.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/train/train_split_1


In [ ]:
# print total char count and total processed lines in the corpus
total_count = 0
for letter, count in counter.most_common():
    total_count += count
print(total_count)
print(processed)

156465
2510


In [ ]:
# print the char report
sum = 0
idx = 0
print('Id\tLetter\tCount\tSum\tPercentile')
for letter, count in counter.most_common():
    sum += count
    percentile = (sum / total_count)
    char_dictionary.add_item(letter)
    idx += 1
    print('%d\t%s\t%7d\t%7d\t%f' % (idx, letter, count, sum, percentile))

print(char_dictionary.item2idx)

Id	Letter	Count	Sum	Percentile
1	 	  30324	  30324	0.193807
2	a	  11394	  41718	0.266628
3	e	   8379	  50097	0.320180
4	ا	   5948	  56045	0.358195
5	l	   5307	  61352	0.392113
6	t	   4913	  66265	0.423513
7	n	   4436	  70701	0.451865
8	ل	   4350	  75051	0.479666
9	o	   4282	  79333	0.507034
10	s	   4226	  83559	0.534043
11	h	   3916	  87475	0.559071
12	ي	   3213	  90688	0.579606
13	m	   3138	  93826	0.599661
14	i	   2980	  96806	0.618707
15	r	   2948	  99754	0.637548
16	d	   2702	 102456	0.654817
17	م	   2561	 105017	0.671185
18	
	   2510	 107527	0.687227
19	y	   2487	 110014	0.703122
20	b	   2328	 112342	0.718001
21	و	   2267	 114609	0.732490
22	ت	   2183	 116792	0.746442
23	ن	   2049	 118841	0.759537
24	w	   1986	 120827	0.772230
25	ب	   1738	 122565	0.783338
26	k	   1694	 124259	0.794165
27	f	   1604	 125863	0.804416
28	ه	   1552	 127415	0.814335
29	ر	   1509	 128924	0.823980
30	ع	   1491	 130415	0.833509
31	3	   1407	 131822	0.842502
32	ك	   1389	 133211	0.851379
33	g	   1276	 1344

In [ ]:
# save the character dictionary to the drive
import pickle
with open('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/char_mappings', 'wb') as f:
    mappings = {
        'idx2item': char_dictionary.idx2item,
        'item2idx': char_dictionary.item2idx
    }
    pickle.dump(mappings, f)

In [ ]:
# load the character dictionary
dictionary = Dictionary.load_from_file('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/char_mappings')

### Forward Embedding

In [ ]:
# training a forward
is_forward_lm = True

# get the corpus, process forward and at the character level
corpus = TextCorpus('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus',
                    dictionary,
                    is_forward_lm,
                    character_level=True)

2020-06-11 08:21:43,252 read text file with 391 lines
2020-06-11 08:21:43,432 read text file with 559 lines


In [ ]:
# instantiate the language model, set hidden size and number of layers
language_model = LanguageModel(dictionary,
                               is_forward_lm,
                               hidden_size=2048,
                               nlayers=1)

In [ ]:
# train the language model
trainer = LanguageModelTrainer(language_model, corpus)

trainer.train('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/language_model_forward',
              sequence_length=250,
              mini_batch_size=100,
              max_epochs=10)

2020-06-11 08:21:43,925 read text file with 1560 lines
2020-06-11 08:21:43,939 shuffled
2020-06-11 08:21:44,583 Sequence length is 250
2020-06-11 08:21:44,589 Split 1	 - (08:21:44)
2020-06-11 08:21:52,769 8 seconds for train split 1
2020-06-11 08:21:53,575 best loss so far 10000.00
2020-06-11 08:21:55,973 ('\nهeeao\U0001f92d0sasaa🚬eاaaaاeayaeel益a4bh<✋eaanصa🙂aa💞aaeaeeea1as\U0001f928aغaa\naoasaeأaanaيa🙏lأر•ineh🙂داa🔥aaaaeaAmت2eب👍aaa😀aeفaalseتp١a🌚اae٠hee🌚aé😘abلو،\U0001f9d0;aa.tكs🤚😃aاl\U0001f975لeaaaae😜aaa😛d💪-mوعas⬇tealوa⚠ؤaجد📉aaaaaاa💞m🚬ل♂😁laيe☑aaاfaرb🤣a٢a\U0001f9d0a😞،aaaseaaatاaن🏽aمi😊لae:sa\n😂aaٍae🙃aa1\u200daa,(eeيaeaeas😱aayw٢y🌍aa|aنg😅aف❤👌ae🌍aاas🍃aaaaي\u200fa👀aaeaeaa2ن👌a💕\u200e%💃eي😁🤝ae💃ae😘\na🤷3d\U0001f970خ🙂♥haلaaataa●ل_naalraخ💕Maaا\U0001f92dيyanleaaاeaaaare🤦a•لa😺(🌍lmaaapb💋aلaaeأrneaا♥لee✋as💞asa,an\nlخنenضaaةab\u200dhadaeetاad٢aaنآأaطaاaaرلlّaeaanaيae\na🔥💋ahaaآaaaaaae•aك🏻b💋o2اoaaوaoiزaelاذ🙄Maeba❤aq😌a❤|nla👌an٤oFaa🖐٠aٍم💃\U0001f92aeaي🏽aeab‘leaL📉\U0001f90daaa;اaaIarلaa🤷a\u2069nfyءa☑ewoaaاe5لn🚬💙

In [ ]:
# check the forward embedding
from flair.embeddings import FlairEmbeddings
from flair.data import Sentence
sentence = Sentence('i’m done')

# init embeddings from your trained LM
char_lm_embeddings_forward = FlairEmbeddings('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/language_model_forward/best-lm.pt')

# embed sentence
char_lm_embeddings_forward.embed(sentence)

# print the embedded tokens.
for token in sentence:
    print(token.text)
    print(token.embedding)

i’m
tensor([ 0.0454, -0.0434,  0.0149,  ..., -0.0013,  0.0161,  0.0493],
       device='cuda:0')
done
tensor([ 0.0220, -0.0454,  0.0140,  ...,  0.0077,  0.0188,  0.0647],
       device='cuda:0')


### Backward Embedding

In [ ]:
# training a backward
is_forward_lm = False

# get the corpus, process backward and at the character level
corpus = TextCorpus('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus',
                    dictionary,
                    is_forward_lm,
                    character_level=True)

2020-06-11 08:23:50,105 read text file with 391 lines
2020-06-11 08:23:50,269 read text file with 559 lines


In [ ]:
# instantiate the language model, set hidden size and number of layers
language_model = LanguageModel(dictionary,
                               is_forward_lm,
                               hidden_size=2048,
                               nlayers=1)

In [ ]:
# train the language model
trainer = LanguageModelTrainer(language_model, corpus)

trainer.train('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/language_model_backward',
              sequence_length=250,
              mini_batch_size=100,
              max_epochs=10)

2020-06-11 08:23:50,751 read text file with 1560 lines
2020-06-11 08:23:50,760 shuffled
2020-06-11 08:23:51,401 Sequence length is 250
2020-06-11 08:23:51,409 Split 1	 - (08:23:51)
2020-06-11 08:23:59,426 8 seconds for train split 1
2020-06-11 08:24:00,249 best loss so far 10000.00
2020-06-11 08:24:02,645 ('taeaeeeبeaeaaaaeاaeaaaeaلاaenealtne#eamoaeebeeaealeaadaenaeaeadnaegنeaeaettaaaaaaeرetaaeeeateaoeeaaaaeaاaoاalاaeetaeاeeaeاaenateeoاaaaaaeلoaeraaaeاaeeaeaaaeaaeeaaeeoae@😭eعoeaaeeeeaaeenes️ااeeeاouااadaeaaeaeaeaenaeee٦ae{n💖aaaeaee6nيپenaaaاeجaeaeaeeaeeeyeaeaeeaaeاaeaaلeاeanteaeلeoeلااeaaeaaeeaaaale‘oadlnealeaaaea؟eaaelateeاrاaaereaaااa-اaeeaاhaaaeeoeaeaeaeaet🏼a\U0001f974aeaeل💞nealaeaeeesنeaاللaeealaeلeaaaاsaennaaeae٧aoaaaeاااetaoeeaeaaaae\n👍eeeaeeaaeetlaaaealaاeeaeاnاaاa•ااaaeaeaoaaaeeاeeeeeetoeteaaaeeeeاeaeeaتaeaeaaaotaa💃اaaa#aeeeaaaaeeeeeeaلeماaeaaلae\neeaeeeeaaيلtاeلoaoaمeaes😘asاeeaseطaمeaaeaetataaheeeeahaaaaaaaee益eaeeaaia●eleaeaeeaateeaeapaeaaaabaaaلeeلelaeaaeebaelateaeleaaaاeheae

In [ ]:
# check the backward embedding
from flair.embeddings import FlairEmbeddings
from flair.data import Sentence
sentence = Sentence('i’m done')

# init embeddings from your trained LM
char_lm_embeddings_backward = FlairEmbeddings('/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/language_model_backward/best-lm.pt')

# embed sentence
char_lm_embeddings_backward.embed(sentence)

# print the embedded tokens.
for token in sentence:
    print(token)
    print(token.embedding)

Token: 1 i’m
tensor([-0.0281, -0.0276, -0.0274,  ..., -0.0105, -0.0450,  0.0039],
       device='cuda:0')
Token: 2 done
tensor([ 3.9812e-03, -1.6066e-02, -2.4749e-02,  ..., -7.0711e-04,
        -3.6896e-02, -9.0107e-05], device='cuda:0')


### Stack the models

In [ ]:
from flair.embeddings import StackedEmbeddings
from flair.data import Sentence
sentence = Sentence('i’m done')

# create a StackedEmbedding object that combines glove and forward/backward flair embeddings
stacked_embeddings = StackedEmbeddings([
                                        char_lm_embeddings_forward,
                                        char_lm_embeddings_backward,
                                       ])

# embed sentence
stacked_embeddings.embed(sentence)

# now check out the embedded tokens.
for token in sentence:
    print(token.text)
    print(token.embedding.tolist())
    print(len(token.embedding.tolist()))

i’m
[0.04544074833393097, -0.04341244325041771, 0.014892029576003551, 0.04106307774782181, 0.08628799021244049, -0.019175639376044273, -0.027688657864928246, 0.026491679251194, -0.0406959168612957, -0.020889032632112503, -0.06856690347194672, 0.013584706000983715, 0.015415409579873085, 0.04760532081127167, -0.09219156205654144, 0.05613285303115845, -0.08870614320039749, -0.10329998284578323, -0.0632043182849884, 0.0737178772687912, -0.06269602477550507, 0.06818314641714096, 0.03320036828517914, 0.04836469143629074, -0.005123985465615988, -0.028177425265312195, 0.05492713674902916, -0.14495396614074707, 0.05737578123807907, 0.08386442065238953, 0.054679274559020996, -0.05310317128896713, -0.015056789852678776, 0.05243536829948425, -0.08175380527973175, -0.04422464221715927, 0.062450774013996124, 0.00850429106503725, 0.09277395904064178, -0.1402408927679062, 0.07748521864414215, 0.004843831993639469, 0.020957889035344124, 0.026349788531661034, 0.02059890329837799, -0.01804976724088192, 0

In [ ]:
# get the embedding vectors for our data
from flair.data import Sentence
file_out = open("/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/pre_trained_flair.aren", 'w')
for file in files:
    print(file)
    with open(file, 'r', encoding='utf-8') as f:
        tokens = 0
        for line in f:
          sentence = Sentence(line)
          stacked_embeddings.embed(sentence)
          for token in sentence:
            output = ""
            output += str(token.text)
            for i in token.embedding.tolist():
              output += " "+str(i)
            output += "\n"
            file_out.writelines(output)

file_out.close() 
print(output)


/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/valid.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/test.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/train/train_split_1
gd3an 0.043213486671447754 -0.0633779838681221 0.0008932820055633783 0.04554362595081329 0.06661678105592728 -0.007652516011148691 -0.06332754343748093 0.055428698658943176 -0.0245596282184124 0.0013904646039009094 -0.07248450815677643 0.052540794014930725 0.024839559569954872 0.04727530479431152 -0.08341695368289948 0.04239818453788757 -0.07776443660259247 -0.09721605479717255 -0.045307546854019165 0.06865517050027847 -0.07537645101547241 0.06445901840925217 0.050487808883190155 0.03105584904551506 -0.006007382646203041 -0.022100426256656647 0.07846179604530334 -0.12287192791700363 0.06062058359384537 0.07418383657932281 0.06916812807321548 -0.05792881175875664 -0.04624498263001442 0.031753022223711014 -0.07427562028169632 -0.05199313908815384 0.057968776673

## Combine Arabic and English Fasttext embedding and Character Embedding

In [ ]:
from flair.embeddings import WordEmbeddings,StackedEmbeddings,CharacterEmbeddings
from flair.data import Sentence

# init embedding
stacked_embeddings = StackedEmbeddings([
                                        WordEmbeddings('en'), # English Wiki and news Fasttext
                                        WordEmbeddings('en-crawl'), # English web crawl Fasttext
                                        WordEmbeddings('ar'), # Arabic Fasttext
                                        CharacterEmbeddings(), # char embedding
                                        ])

# create a sentence
sentence = Sentence('الحياة gamela gedan we now its agmal')

# embed words in sentence
stacked_embeddings.embed(sentence)

for token in sentence:
    emb = token.embedding.tolist()
    print(token.text)
    print(emb)
    print(len(emb))

/usr/local/lib/python3.6/dist-packages/smart_open/smart_open_lib.py:253: UserWarning: This function is deprecated, use smart_open.open instead. See the migration notes for details: https://github.com/RaRe-Technologies/smart_open/blob/master/README.rst#migrating-to-the-new-open-function
  'See the migration notes for details: %s' % _MIGRATION_NOTES_URL


الحياة
[-0.05829999968409538, -0.061900001019239426, -0.044599998742341995, 0.1298999935388565, 0.11460000276565552, 0.05299999937415123, 0.03290000185370445, 0.01119999960064888, 0.0034000000450760126, 0.14990000426769257, -0.08980000019073486, 0.014999999664723873, 0.02800000086426735, -0.0, -0.030400000512599945, 0.05469999834895134, 0.07609999924898148, 0.018200000748038292, 0.029600000008940697, 0.02850000001490116, -0.1956000030040741, 0.08340000361204147, -0.08579999953508377, -0.08009999990463257, 0.003100000089034438, 0.019099999219179153, 0.1468999981880188, 0.004900000058114529, 0.050999999046325684, 0.12839999794960022, 0.031099999323487282, 0.03880000114440918, 0.1535000056028366, 0.04170000180602074, 0.046799998730421066, 0.06340000033378601, -0.03889999911189079, -0.0005000000237487257, -0.12319999933242798, 0.06539999693632126, 0.186599999666214, -0.051500000059604645, -0.013700000010430813, 0.03790000081062317, 0.008700000122189522, 0.007000000216066837, 0.004600000102

In [ ]:
# get the embedding vectors for our data
from flair.data import Sentence
file_out = open("/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/pre_trained_fasttext.aren", 'w')
for file in files:
    print(file)
    with open(file, 'r', encoding='utf-8') as f:
        k = 0
        j = 0
        for line in f:
          k = k+1
          sentence = Sentence(line)
          if len(sentence) != 0:
            j = j+1
            stacked_embeddings.embed(sentence)
            for token in sentence:
              output = ""
              output += str(token.text)
              for i in token.embedding.tolist():
                output += " "+str(i)
              output += "\n"
              file_out.writelines(output)

file_out.close() 
print(output)
print(k,j)


/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/valid.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/test.txt
/content/gdrive/My Drive/Colab Notebooks/Flair Embedding/corpus/train/train_split_1
gd3an 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0